# Stage 8 — Text Generation

Load the trained checkpoint and sample from it three ways:

1. **Greedy** — always pick `argmax`. Deterministic, often repetitive.
2. **Temperature** — divide logits by `T` before softmax. `T < 1` sharpens (more deterministic), `T > 1` flattens (more random).
3. **Top-k** — restrict the distribution to the `k` most likely tokens, renormalize, then sample.

We also add a tiny **repetition guard** (token-level) to demonstrate one common knob beyond top-k.

In [1]:
from pathlib import Path

import torch
import torch.nn.functional as F
import tiktoken

from model import GPTModel

torch.manual_seed(123)
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)

device: mps


## Load the trained checkpoint

In [2]:
ckpt = torch.load(Path('checkpoints/mahabharata_gpt.pt'), map_location=device, weights_only=True)
CFG  = ckpt['cfg']
print('cfg:', CFG)

model = GPTModel(CFG).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

tok = tiktoken.get_encoding('gpt2')
print('model loaded — params:', sum(p.numel() for p in model.parameters()))

cfg: {'vocab_size': 50257, 'context_len': 128, 'emb_dim': 256, 'n_heads': 8, 'n_layers': 4, 'drop_rate': 0.1, 'qkv_bias': False}
model loaded — params: 28920832


## One sampler, three knobs

All three strategies are a single function — pick `temperature=0` for greedy, `top_k=None` for pure-temperature, both set for top-k sampling.

In [3]:
@torch.no_grad()
def generate(
    prompt: str,
    max_new_tokens: int = 100,
    temperature: float = 1.0,
    top_k: int | None = None,
    repetition_penalty: float = 1.0,
    eos_id: int | None = None,
) -> str:
    """Generate text from `prompt`.

    - temperature=0 → greedy argmax (deterministic)
    - top_k=k       → restrict sampling to the k highest-prob tokens
    - repetition_penalty>1 → divides logits of already-generated tokens (lower p of repeating)
    """
    ids = torch.tensor([tok.encode(prompt)], dtype=torch.long, device=device)
    ctx_len = CFG['context_len']

    for _ in range(max_new_tokens):
        ctx = ids[:, -ctx_len:]
        logits = model(ctx)[:, -1, :]  # (1, vocab)

        if repetition_penalty != 1.0:
            seen = ids[0].unique()
            logits[0, seen] = logits[0, seen] / repetition_penalty

        if temperature == 0:
            next_id = logits.argmax(dim=-1, keepdim=True)
        else:
            logits = logits / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, k=min(top_k, logits.size(-1)))
                # mask everything below the k-th value
                logits = torch.where(
                    logits < v[:, [-1]],
                    torch.full_like(logits, float('-inf')),
                    logits,
                )
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        if eos_id is not None and next_id.item() == eos_id:
            break
        ids = torch.cat([ids, next_id], dim=1)

    return tok.decode(ids[0].tolist())

## Side-by-side comparison

In [4]:
PROMPT = 'Arjuna said,'
MAX_NEW = 120

def banner(label):
    print('\n' + '=' * 78)
    print(label)
    print('=' * 78)

banner('1) Greedy (temperature=0)')
print(generate(PROMPT, MAX_NEW, temperature=0))

banner('2) Temperature=1.0, no top-k (raw softmax sampling)')
torch.manual_seed(0)
print(generate(PROMPT, MAX_NEW, temperature=1.0))

banner('3) Temperature=0.8, top-k=40 (typical GPT-style)')
torch.manual_seed(0)
print(generate(PROMPT, MAX_NEW, temperature=0.8, top_k=40))

banner('4) Temperature=0.8, top-k=40, repetition_penalty=1.2')
torch.manual_seed(0)
print(generate(PROMPT, MAX_NEW, temperature=0.8, top_k=40, repetition_penalty=1.2))


1) Greedy (temperature=0)
Arjuna said, "O son of the Pandavas, I will be slain by the Pandavas. I will be able to be able to be able to be slain. I will be able to be able to be able to be able to the Pandavas. I will be able to the Pandavas. I will be able to the Pandavas. I will be able to the Pandavas, I will be able to the Pandavas. I will be able to the Pandavas, I will be able to the Pandavas. I will be able to the Pandav

2) Temperature=1.0, no top-k (raw softmax sampling)
Arjuna said, and how shall administered service happily. Those that I have been killed in this universe, then divisions, I shall never feel your son urged."
"When you can want a defeat the heavenly den! O King  Balarman's house, and then cannot can kill him who has ever finished to the kingdom Arjuna, the other orderful increasedlust?
 fragments and slay a moment felt will tell Arjuna, leading the causes of seven Grandsire our brother replied to see me after the encountered request, and then Nagala compassion

## Temperature sweep on one prompt

See how `temperature` shifts the determinism/creativity dial.

In [5]:
for T in (0.5, 0.8, 1.0, 1.3):
    torch.manual_seed(42)
    banner(f'temperature = {T}')
    print(generate('On the field of Kurukshetra,', max_new_tokens=80, temperature=T, top_k=40))


temperature = 0.5
On the field of Kurukshetra, and Karna.


The Pandavas, the Pandavas, the King Yudhisthira's son, the Pandavas, the Kritavasena, Bhima and the Pandavas. Bhima then entered the Pandavas and the city of the Pandavas with the Pandavas. He then took up a great warrior.




temperature = 0.8
On the field of Kurukshetra, and Kritavasena with a great King Virata. The Pandavasena was a son of the King's son. The Pandavas, the Kritarman came to the forest and was like a matter. Bhima then entered the battlefield. In the great mace. Satyaki with the son had sent the Pandava army, he was also also told his own

temperature = 1.0
On the field of Kurukshetra, and Kritavrata with a great King Virata. The sound of a matter was a son like a certain brahmana, and then spoke the Kritarmanmana, they are equal to fight with the end of a fierce kingdom of my army and the city. They now travel from the sun and he was not move. Lord, he was also also told his own

temperature = 1.3
On the 

## Your turn

Tweak the prompts below. Good Mahabharata starters: names (`Bhishma`, `Yudhishthira`, `Krishna`), scenes (`On the battlefield,`), or speech acts (`The sage said,`).

In [6]:
torch.manual_seed(7)
print(generate(
    'Bhishma, lying on the bed of arrows, spoke to',
    max_new_tokens=150,
    temperature=0.8,
    top_k=40,
    repetition_penalty=1.1,
))

Bhishma, lying on the bed of arrows, spoke to be a maces. If he was the field of his father's death. When the enemy released one who was in the Lord Krishna then killed the Supreme Personality of God. He is more by good qualities of fear of his prowess and thus he said, and after Kaurava army. With the forest then ordered Susharman, Yudhisthira then I will have never seen that I am known as they has now come to us? One who are you are so not listen to be given a brahira, who do not be killed the Pandavas. He should be performed. O Arjuna again has taken shelter of the battle, in this, which my sons of the Pandavas were not attain to me


## What you've built

End-to-end, from scratch:

| Stage | What |
| ----- | ---- |
| 1 | Env, deps, device, seed |
| 2 | Mahabharata corpus from Kaggle |
| 3 | Cleaned, normalized, split → `train.txt`/`val.txt` |
| 4 | BPE tokenization (from scratch + tiktoken) → `train.bin`/`val.bin` |
| 5 | Attention: scaled dot-product → causal → multi-head |
| 6 | LayerNorm, GELU, FFN, full TransformerBlock |
| 7 | Stack into GPT, train with AdamW + grad clip, save checkpoint |
| 8 | Sampling: greedy, temperature, top-k, repetition penalty |

Natural extensions:

- **Bigger model / more data**: bump `emb_dim`/`n_layers`, train longer.
- **Mixed precision** (`torch.amp.autocast`) and `torch.compile` for speed.
- **Cosine LR schedule** with warmup.
- **Better sampling**: nucleus (top-p), beam search.
- **Instruction-tune / fine-tune** on Q&A pairs.
- **Evaluation**: perplexity on a held-out test slice.